<a href="https://colab.research.google.com/github/UTD2026/Mixed_Dataset_Testing_STA/blob/main/Job1_lr1e6_gsm8k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Job 1 — lr-1e-6 gsm8k counterfactual (issue #5)

Standalone / clean-kernel runner. Decides #1: the gsm8k top-CE adapters behind "every routing
policy is flat on gsm8k" were trained at lr 5e-5; upstream TLM uses **1e-6** on reasoning benches.
If the adapter was just trained too hot, "CE gates nothing for CoT" may really be "there was no
usable adapter to route to." This retrains the *same* top-CE selection at 1e-6 and re-reads routing.

**Baked-in guard (today's lesson):** gsm8k grading has a ~0.17 answer-extraction band (math-mode /
last-number / first-number disagree by that much on long CoT). Job 1's readout is "did routing
*move*" — so any routing delta **smaller than the grader band is noise, not signal**. This notebook
generates at 512 tokens (no truncation), grades 3 ways, and refuses to call movement inside the band.

Scope tonight: **2B only** (where the headroom counter-evidence lives: base 71.3 / oracle 78.4).
The 0.8B "completeness" arm is secondary — leave it for a fresh session.

## 0 · Knobs

In [17]:
REPO_URL="https://github.com/UTD2026/rishabh-tlm.git"
WORK="/content/tlm_job1"; LOCAL_REPO_ZIP="/content/rishabh-tlm-main.zip"; GITHUB_TOKEN_ENV="GITHUB_TOKEN"
MODEL="Qwen/Qwen3.5-2B"; MTAG="2b"
N=5000; GEN_BATCH=16; MAXNEW=512
RUN_GPU=True
# where the cesplit-sweep-2026-07-02 release unpacks to (sel_topce.json / features.jsonl / preds_base.jsonl):
SWEEP_REL="cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k"
print({"model":MODEL,"N":N,"maxnew":MAXNEW})

{'model': 'Qwen/Qwen3.5-2B', 'N': 5000, 'maxnew': 512}


## 1 · Env (Colab GPU, HF-only)

In [10]:
import os,sys,subprocess
from pathlib import Path
Path(WORK).mkdir(parents=True,exist_ok=True); os.chdir(WORK)
def pip(*a): print("pip",*a); subprocess.check_call([sys.executable,"-m","pip",*a])
if RUN_GPU:
    pip("install","-U","peft","accelerate","scipy","sympy","ninja","huggingface_hub")
    if "Qwen3.5" in MODEL: pip("install","-U","transformers[serving] @ git+https://github.com/huggingface/transformers.git@main")
    for p in ["torchaudio","torchvision","torchao"]:
        try: pip("uninstall","-y",p)
        except Exception as e: print("skip",p,e)
    for m in list(sys.modules):
        if m.startswith(("transformers","peft","torchaudio","torchvision","torchao")): del sys.modules[m]
import numpy as np, pandas as pd
print("ready")

pip install -U peft accelerate scipy sympy ninja huggingface_hub
pip install -U transformers[serving] @ git+https://github.com/huggingface/transformers.git@main
pip uninstall -y torchaudio
pip uninstall -y torchvision
pip uninstall -y torchao
ready


## 2 · Clone / upload repo (with the cesplit-sweep release assets)

In [15]:
import glob
hits = glob.glob("/content/**/cesplit_sweep_2026-07-02.zip", recursive=True) + \
       glob.glob(f"{REPO_DIR}/**/cesplit_sweep_2026-07-02.zip", recursive=True)
print(hits)

['/content/tlm_job1/cesplit_sweep_2026-07-02.zip']


In [16]:
import zipfile, glob
z = hits[0]
with zipfile.ZipFile(z) as zf:
    zf.extractall(REPO_DIR)
print("gsm8k files:", glob.glob(f"{REPO_DIR}/**/qwen3_5-2b/gsm8k/*", recursive=True))
print("sel files:  ", glob.glob(f"{REPO_DIR}/**/sel_topce*", recursive=True)[:5])

gsm8k files: ['/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/preds_base.jsonl', '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/preds_adapted_top10.jsonl', '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/sel_top10.json', '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/grid_top10.json', '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/features.jsonl', '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/adapter_top10']
sel files:   []


In [11]:
from google.colab import files
up = files.upload()   # pick cesplit_sweep_2026-07-02.zip
import zipfile
name = [n for n in up if n.endswith(".zip")][0]
with zipfile.ZipFile(name) as zf: zf.extractall(REPO_DIR)

Saving cesplit_sweep_2026-07-02.zip to cesplit_sweep_2026-07-02.zip


In [14]:
import zipfile, glob, os
z = "cesplit_sweep_2026-07-02.zip"                      # in cwd from the upload
z = z if os.path.exists(z) else f"/content/{z}"
with zipfile.ZipFile(z) as zf:
    names = zf.namelist()
    zf.extractall(REPO_DIR)
print("extracted", len(names), "entries")
# what do the gsm8k inputs actually look like?
print("gsm8k files:", glob.glob(f"{REPO_DIR}/**/qwen3_5-2b/gsm8k/*", recursive=True))
print("any sel_topce:", glob.glob(f"{REPO_DIR}/**/sel_topce*", recursive=True)[:5])

FileNotFoundError: [Errno 2] No such file or directory: '/content/cesplit_sweep_2026-07-02.zip'

In [18]:
import os,subprocess,shutil,zipfile,glob
from pathlib import Path
repo_dir=Path(WORK)/"rishabh-tlm"
def _clone(u):
    tok=os.environ.get(GITHUB_TOKEN_ENV,"").strip()
    uu=u.replace("https://",f"https://{tok}@",1) if (tok and u.startswith("https://github.com/")) else u
    return subprocess.run(["git","clone","--depth","1",uu,str(repo_dir)],text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
def _extract(zp):
    zp=Path(zp).expanduser(); assert zp.exists(),f"no zip {zp}"
    tmp=Path(WORK)/"_zip"; shutil.rmtree(tmp,ignore_errors=True); tmp.mkdir(parents=True)
    with zipfile.ZipFile(zp) as zf: zf.extractall(tmp)
    cand=tmp if (tmp/"cuda_ttl").exists() else next((p for p in tmp.rglob("*") if p.is_dir() and (p/"cuda_ttl").exists()),None)
    assert cand,"repo root not found"; shutil.rmtree(repo_dir,ignore_errors=True); shutil.move(str(cand),str(repo_dir))
if not repo_dir.exists():
    if _clone(REPO_URL).returncode!=0:
        cands=glob.glob(f"{WORK}/*.zip")+glob.glob("/content/*.zip")+([LOCAL_REPO_ZIP] if Path(LOCAL_REPO_ZIP).exists() else [])
        if cands: _extract(cands[0])
        else:
            from google.colab import files; up=files.upload()
            _extract(Path(os.getcwd())/([n for n in up if n.endswith(".zip")] or list(up))[0])
os.chdir(repo_dir); REPO_DIR=Path.cwd(); print("repo:",REPO_DIR)

# ---- REQUIRED release inputs (cesplit-sweep-2026-07-02). Fail loud with the exact fix. ----
SWEEP=REPO_DIR/SWEEP_REL
need={"sel":SWEEP/"sel_top10.json","feat":SWEEP/"features.jsonl","base":SWEEP/"preds_base.jsonl"}
missing=[k for k,p in need.items() if not p.exists()]
if missing:
    raise FileNotFoundError(
      f"Missing Job-1 release inputs {missing} under {SWEEP}.\n"
      f"Download the cesplit-sweep-2026-07-02 release into the repo root, e.g.:\n"
      f"  gh release download cesplit-sweep-2026-07-02 -p '*.zip' && unzip cesplit_sweep_2026-07-02.zip -d {REPO_DIR}\n"
      f"(the zip mirrors sweep2/<mtag>/<dataset>/).")
print("release inputs present:", {k:str(p) for k,p in need.items()})

repo: /content/tlm_job1/rishabh-tlm
release inputs present: {'sel': '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/sel_top10.json', 'feat': '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/features.jsonl', 'base': '/content/tlm_job1/rishabh-tlm/cesplit_sweep_2026-07-02/sweep2/qwen3_5-2b/gsm8k/preds_base.jsonl'}


## 3 · Helpers (model I/O, grading, subprocess streamer)

In [19]:
import torch, json, transformers, re
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm.auto import tqdm
SCRIPTS=REPO_DIR/"cuda_ttl/ab_routing"; sys.path.insert(0,str(SCRIPTS)); import grading

def load_lm(path):
    tok=AutoTokenizer.from_pretrained(path,trust_remote_code=True)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    tok.padding_side="left"
    cfg=AutoConfig.from_pretrained(path,trust_remote_code=True)
    arch=(getattr(cfg,"architectures",None) or [""])[0]; cls=getattr(transformers,arch,None) or AutoModelForCausalLM
    return cls.from_pretrained(path,torch_dtype=torch.bfloat16,device_map="cuda").eval(), tok
def wrap(tok,q):
    m=[{"role":"user","content":q}]
    try: return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True)
def load_adapter(base_path, adir):
    b,t=load_lm(base_path); return PeftModel.from_pretrained(b,str(adir)).eval(), t
def qof(r): return (r.get("question") or r.get("instruction") or "").strip()
def goldof(r):
    for k in ("output","answer","answers"):
        if r.get(k) not in (None,""): return str(r[k]).strip()
    return ""
def sh(cmd,log,label):
    import time; print(f"  RUN {label}",flush=True); t=time.time()
    with open(log,"w") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for i,ln in enumerate(p.stdout):
            f.write(ln)
            if i%20==0: print(f"    …{label} {time.time()-t:0.0f}s",flush=True)
        if p.wait()!=0: raise RuntimeError(f"failed {cmd} (see {log})")
    print(f"  {label} done {time.time()-t:0.0f}s",flush=True)

def hf_greedy(model,tok,recs,idxs,max_new=512,desc="gen"):
    preds={}; ansids={}; side=tok.padding_side; tok.padding_side="left"
    for s in tqdm(range(0,len(idxs),GEN_BATCH),desc=desc,leave=False):
        ch=idxs[s:s+GEN_BATCH]
        enc=tok([wrap(tok,qof(recs[i])) for i in ch],return_tensors="pt",padding=True,truncation=True,max_length=2048).to(model.device)
        w=enc["input_ids"].shape[1]
        with torch.inference_mode():
            o=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.pad_token_id)
        seq=o[:,w:]; mask=(seq!=tok.pad_token_id); dec=tok.batch_decode(seq,skip_special_tokens=True)
        for bi,i in enumerate(ch): preds[i]=dec[bi]; ansids[i]=seq[bi][mask[bi]].tolist()
    tok.padding_side=side; return preds, ansids
print("helpers ready")

helpers ready


## 4 · Retrain @ 1e-6 · generate @ 512 · grade 3 ways

The 3-way grade is the guard: `math` (repo math grader, handles `$150$`/LaTeX), `lastnum` (last
number / after an answer cue), `firstnum` (repo numeric grader = the biased default). The spread
between them is the **grader band** — the noise floor Job 1's routing delta must clear.

In [20]:
DATA=str(REPO_DIR/"data/AdaptEval/gsm8k_random_5k.json")
J1=Path(WORK)/"job1"/MTAG; J1.mkdir(parents=True,exist_ok=True)

def _goldg(raw):
    m=re.search(r'####\s*(.+)',raw); return (m.group(1).strip() if m else raw)
def _lastnum(t):
    cues=re.findall(r'(?:answer|conclusion|total|=)\s*[:*\s$]*(-?[\d,]+(?:\.\d+)?)',t,re.I)
    nums=re.findall(r'-?[\d,]+(?:\.\d+)?',t.replace('$',''))
    p=(cues[-1] if cues else (nums[-1] if nums else None)); return re.sub(',','',p) if p else None
def grade3(pred,gold_raw):
    g=_goldg(gold_raw); o={}
    try: o["math"]=grading.is_correct(grading.extract(pred,"math").value,g,"math")
    except: o["math"]=False
    try:
        pv=_lastnum(pred); o["lastnum"]=(pv is not None and abs(float(pv)-float(re.sub(r'[%,$\s]','',g)))<=1e-3)
    except: o["lastnum"]=False
    o["firstnum"]=grading.is_correct(grading.extract(pred,"numeric").value,g,"numeric")
    return o

if RUN_GPU:
    adir=J1/"adapter"
    if not (adir/"adapter_model.safetensors").exists():
        sh([sys.executable,str(SCRIPTS/"ab_train_ttl.py"),"--model",MODEL,"--data",DATA,
            "--selection-file",str(need["sel"]),"--max-samples",str(N),
            "--learning-rate","1e-6","--output-dir",str(adir),"--no-merge"], J1/"train.log","train@1e-6")
    recs=json.load(open(DATA))[:N]; idxs=list(range(len(recs)))
    am,atok=load_adapter(MODEL,adir)
    nz=sum(1 for n,p in am.named_parameters() if "lora_B" in n and p.abs().max()>0); assert nz>0,"adapter no-op"
    apred,aids=hf_greedy(am,atok,recs,idxs,max_new=MAXNEW,desc=f"{MTAG} adapted@1e-6")
    del am; torch.cuda.empty_cache()
    trunc=np.mean([len(aids[i])>=MAXNEW-1 and not apred[i].strip().endswith((".","!","?")) for i in idxs])
    bpred={json.loads(l)["idx"]:json.loads(l).get("predict","") for l in open(need["base"])}
    Bok={m:[] for m in ("math","lastnum","firstnum")}; Aok={m:[] for m in ("math","lastnum","firstnum")}
    for i in idxs:
        bg=grade3(bpred.get(i,""),goldof(recs[i])); ag=grade3(apred[i],goldof(recs[i]))
        for m in Bok: Bok[m].append(bg[m]); Aok[m].append(ag[m])
    print(f"\nadapted trunc@{MAXNEW}={trunc:.3f}  (want <0.05)")
    rows=[];
    for m in ("math","lastnum","firstnum"):
        b=np.mean(Bok[m]); a=np.mean(Aok[m]); orc=np.mean([Bok[m][k] or Aok[m][k] for k in range(len(idxs))])
        rows.append(dict(grader=m,base=round(b,3),adapt=round(a,3),headroom=round(orc-b,3)))
    G=pd.DataFrame(rows); display(G)
    GRADER_BAND=float(G.base.max()-G.base.min())
    with open(J1/"preds_adapted_1e6.jsonl","w") as f:
        for i in idxs: f.write(json.dumps({"idx":i,"predict":apred[i],"label":_goldg(goldof(recs[i]))})+"\n")
    print(f"grader band = {GRADER_BAND:.3f}  -> routing deltas smaller than this are NOISE")
else:
    print("RUN_GPU=False")

  RUN train@1e-6
    …train@1e-6 4s
  train@1e-6 done 49s


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

2b adapted@1e-6:   0%|          | 0/313 [00:00<?, ?it/s]


adapted trunc@512=0.111  (want <0.05)


,grader,base,adapt,headroom
0,math,0.106,0.551,0.455
1,lastnum,0.126,0.634,0.519
2,firstnum,0.143,0.694,0.567


grader band = 0.037  -> routing deltas smaller than this are NOISE


## 5 · Official routing grid + verdict

Runs the spec's `ab_route_grid` on the clean 1e-6 adapted preds, then compares the best routing
policy against all-base — and against the grader band. `curves` are lists of `{y,acc,...}`.

In [22]:
if RUN_GPU:
    grid=json.load(open(J1/"grid_gsm8k_1e6.json")); refs=grid.get("refs",{}); curves=grid.get("curves",{})
    print("refs:",{k:(round(v,3) if isinstance(v,(int,float)) else v) for k,v in refs.items()})
    all_base=refs.get("all_base") or refs.get("base")
    if isinstance(all_base,dict):                     # some schemas nest {"acc":..,"n":..}
        all_base=all_base.get("acc") or all_base.get("value")
    deltas={}
    for name,cur in curves.items():
        if not cur: continue
        best=max(cur,key=lambda p:p["acc"]); deltas[name]=best["acc"]-all_base
        print(f"  {name:9s} best acc={best['acc']:.3f} @Y={best['y']}  delta_vs_base={best['acc']-all_base:+.3f}")
    max_delta=max(deltas.values()) if deltas else 0.0
    print("\n=== JOB 1 VERDICT (2B) ===")
    print(f"grader band = {GRADER_BAND:.3f}   |   best routing delta = {max_delta:+.3f}")
    if max_delta > GRADER_BAND:
        print("-> ROUTING MOVED on the 1e-6 adapter (delta exceeds grader band); #1 framing needs revision.")
    else:
        print("-> routing delta WITHIN grader band: INCONCLUSIVE on gsm8k (extraction noise dominates).")

refs: {'all_base': 0.713, 'all_adapted': 0.667, 'oracle': 0.743, 'tau_0.3': 0.667, 'tau_0.5': 0.667, 'iso_ce_m200': {'acc_mean': 0.7071200000000001, 'acc_sd': 0.007778534566356298, 'base_frac_mean': 0.8766799999999999}}
  pct       best acc=0.713 @Y=100  delta_vs_base=+0.000
  v2stream  best acc=0.714 @Y=75  delta_vs_base=+0.001

=== JOB 1 VERDICT (2B) ===
grader band = 0.037   |   best routing delta = +0.001
-> routing delta WITHIN grader band: INCONCLUSIVE on gsm8k (extraction noise dominates).


In [25]:
print(json.loads(open(need["base"]).readline())["predict"][:200])

To find the total amount of money Stefan's family spends, we need to calculate the cost of the food and then add the tip.

### Step 1: Calculate the cost of the appetizer
The appetizer costs **$10**.



In [27]:
MAXNEW = 768
recs=json.load(open(DATA))[:N]; idxs=list(range(len(recs)))
# base — regenerate at 768 (don't use the release preds_base for grading; they're truncated)
bm,btok=load_lm(MODEL)
bpred,baids=hf_greedy(bm,btok,recs,idxs,max_new=MAXNEW,desc="base@768")
del bm; torch.cuda.empty_cache()
# adapted — 1e-6 adapter, same budget
am,atok=load_adapter(MODEL, J1/"adapter")
apred,aaids=hf_greedy(am,atok,recs,idxs,max_new=MAXNEW,desc="adapted@768")
del am; torch.cuda.empty_cache()
tb=np.mean([len(baids[i])>=MAXNEW-1 and not bpred[i].strip().endswith((".","!","?")) for i in idxs])
ta=np.mean([len(aaids[i])>=MAXNEW-1 and not apred[i].strip().endswith((".","!","?")) for i in idxs])
print(f"trunc base={tb:.3f} adapted={ta:.3f} (want <0.05)")
# grade both with math-mode (the reliable one for CoT), write clean pred files for the grid
def ok(p,g):
    try: return grading.is_correct(grading.extract(p,"math").value,_goldg(goldof(g)),"math")
    except: return False
bb=np.mean([ok(bpred[i],recs[i]) for i in idxs]); aa=np.mean([ok(apred[i],recs[i]) for i in idxs])
print(f"base_acc={bb:.3f}  adapt_acc={aa:.3f}")
for tag,preds in [("base",bpred),("adapted",apred)]:
    with open(J1/f"preds_{tag}_clean.jsonl","w") as f:
        for i in idxs: f.write(json.dumps({"idx":i,"predict":preds[i],"label":_goldg(goldof(recs[i]))})+"\n")

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

base@768:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

adapted@768:   0%|          | 0/313 [00:00<?, ?it/s]

trunc base=0.043 adapted=0.044 (want <0.05)
base_acc=0.563  adapt_acc=0.563


In [28]:
sh([sys.executable,str(SCRIPTS/"ab_route_grid.py"),"--dataset","gsm8k",
    "--features",str(need["feat"]),"--base-preds",str(J1/"preds_base_clean.jsonl"),
    "--adapted-preds",str(J1/"preds_adapted_clean.jsonl"),"--out",str(J1/"grid_clean.json")], J1/"grid.log","grid")

  RUN grid
    …grid 1s
  grid done 1s


In [29]:
sh([sys.executable,str(SCRIPTS/"ab_route_grid.py"),"--dataset","gsm8k",
    "--features",str(need["feat"]),"--base-preds",str(J1/"preds_base_clean.jsonl"),
    "--adapted-preds",str(J1/"preds_adapted_clean.jsonl"),"--out",str(J1/"grid_clean.json")], J1/"grid.log","grid")
sh([sys.executable,str(REPO_DIR/"teams/rishabh/sbar_phase0.py"),"--grid",str(J1/"grid_clean.json")], J1/"phase0.log","phase0")
print(open(J1/"phase0.log").read()[-1500:])

  RUN grid
    …grid 1s
  grid done 1s
  RUN phase0
    …phase0 0s


RuntimeError: failed ['/usr/bin/python3', '/content/tlm_job1/rishabh-tlm/teams/rishabh/sbar_phase0.py', '--grid', '/content/tlm_job1/job1/2b/grid_clean.json'] (see /content/tlm_job1/job1/2b/phase0.log)

In [31]:
g=json.load(open(J1/"grid_clean.json"))
print("refs:", {k:(round(v,3) if isinstance(v,(int,float)) else v) for k,v in g["refs"].items()})
for name,cur in g["curves"].items():
    if cur:
        best=max(cur,key=lambda p:p["acc"])
        print(f"  {name:9s} best={best['acc']:.3f} @Y={best['y']}  delta={best['acc']-g['refs']['all_base']:+.3f}")

refs: {'all_base': 0.701, 'all_adapted': 0.705, 'oracle': 0.74, 'tau_0.3': 0.705, 'tau_0.5': 0.705, 'iso_ce_m200': {'acc_mean': 0.7027599999999999, 'acc_sd': 0.002731739372634222, 'base_frac_mean': 0.35014}}
  pct       best=0.705 @Y=0  delta=+0.004
  v2stream  best=0.705 @Y=0  delta=+0.004
